<a href="https://colab.research.google.com/github/ziyiWANGG/7AAVDH26_001_Final_Coding_Project/blob/main/CT_Denoising_DCT_vs_Gaussian.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Brain CT Image Denoising: DCT vs Gaussian Filtering**


---


## Project Overview
This project compares the denoising performance of DCT-based frequency domain filtering and Gaussian spatial domain filtering on low-dose brain CT images.


---


## Key Methods
1. **Denoising Methods**
   - DCT Denoising: 8×8 block DCT + JPEG quantization table (scipy.fft.dctn, idctn, quantization rounding)
   - Gaussian Filtering: Weighted average with Gaussian kernel (skimage.filters.gaussian)

2. **Evaluation Metrics**
   - PSNR (Peak Signal-to-Noise Ratio)
   - SSIM (Structural Similarity Index)

3. **Visualization**
   - Image comparison
   - PSNR/SSIM bar chart


---


## Data Source

The brain CT dataset used in this study is obtained from the PhysioNet repository:

Hssayeni, M. (2020). Computed Tomography Images for Intracranial Hemorrhage Detection and Segmentation. PhysioNet. https://physionet.org/content/ct-ich/1.3.1/

## **Workspace**

In [ ]:
pip install numpy matplotlib scipy scikit-image opencv-python pillow

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from skimage import io, filters, metrics
from scipy.fft import dctn, idctn
from skimage.util import img_as_float
import warnings

In [ ]:
# Clone the repository
!git clone https://github.com/ziyiWANGG/7AAVDH26_001_Final_Coding_Project.git

import os
os.chdir('/content/7AAVDH26_001_Final_Coding_Project')
print(f"Current working directory: {os.getcwd()}")

In [ ]:
# Collect image paths
base_path = "./CT_images"
folders = ["054_brain", "079_brain"]
image_paths = []

for folder in folders:
    folder_path = os.path.join(base_path, folder)
    if os.path.exists(folder_path):
        for fname in os.listdir(folder_path):
            if fname.lower().endswith('.jpg'):
                image_paths.append(os.path.join(folder_path, fname))
        print(f"{folder}: Found {len([f for f in os.listdir(folder_path) if f.lower().endswith('.jpg')])} image(s)")
print(f"Total found: {len(image_paths)} image(s)")

## **CT Denoising**

**Step1**. DCT Denoising

In [ ]:
# Denoising a grayscale image using block-based DCT + quantization + inverse quantization + IDCT
def dct_denoise(image, quality_factor=1.0):
    """
    image: 2D NumPy array, values ​​range 0~255
    quality_factor: Quantization scaling factor (<1 enhances denoising, >1 preserves more details)
    """

    ## JPEG standard luminance quantisation table
    Q_table = np.array([
        [16, 11, 10, 16, 24, 40, 51, 61],
        [12, 12, 14, 19, 26, 58, 60, 55],
        [14, 13, 16, 24, 40, 57, 69, 56],
        [14, 17, 22, 29, 51, 87, 80, 62],
        [18, 22, 37, 56, 68, 109, 103, 77],
        [24, 35, 55, 64, 81, 104, 113, 92],
        [49, 64, 78, 87, 103, 121, 120, 101],
        [72, 92, 95, 98, 112, 100, 103, 99]
    ])
    Q_table = Q_table * quality_factor

    ## Fill the edges to a multiple of 8.
    h, w = image.shape
    pad_h = (8 - h % 8) % 8
    pad_w = (8 - w % 8) % 8
    img_pad = np.pad(image, ((0, pad_h), (0, pad_w)), mode='edge')
    img_pad = img_pad.astype(np.float32)

    denoised_pad = np.zeros_like(img_pad)
    """
    Note:
    For the part about filling the edges, I used Deepseek to help me complete the code. The prompt is as follows:
    I am writing a DCT-based image denoising function that needs to handle grayscale images of arbitrary size. Please help me implement the following:
    1. Calculate the number of rows and columns to pad so that the image dimensions become multiples of 8 (required for JPEG blocking)
    2. Create a zero array of the same size as the padded image to store the denoised result
    Libraries to use: numpy and scipy.fft
    """

    ## Block processing
    """
    Because the JPEG standard requires block-by-block processing for larger images,
    two nested loops must be written manually to iterate through all blocks.
    """
    for i in range(0, img_pad.shape[0], 8):
        for j in range(0, img_pad.shape[1], 8):
            block = img_pad[i:i+8, j:j+8]
            ### Centre/centralise (shift by 128)
            block_centered = block - 128
            ### DCT
            dct_block = dctn(block_centered, norm='ortho')
            ### Quantisation
            quantized = np.round(dct_block / Q_table)
            ### Inverse quantisation
            dequantized = quantized * Q_table
            ### IDCT
            idct_block = idctn(dequantized, norm='ortho')
            ### Inverse centre/centralise shift
            denoised_pad[i:i+8, j:j+8] = idct_block + 128

    ## Crop back to original dimensions
    denoised = np.clip(denoised_pad[:h, :w], 0, 255).astype(np.uint8)
    return denoised

**Step2**. Gaussian Filter Denoising

In [ ]:
def gaussian_denoise(image, sigma=1.0):
    """Gaussian filtering denoising, sigma controls smoothing strength"""
    image_float = img_as_float(image)  # convert to [0,1] float
    denoised_float = filters.gaussian(image_float, sigma=sigma, preserve_range=False)
    denoised = (denoised_float * 255).astype(np.uint8)
    return denoised

## **Evaluation**

**Step1**. Calculate PSNR & SSIM

In [ ]:
# Calculate Peak Signal-to-Noise Ratio (PSNR)
def calculate_psnr(img_orig, img_denoised):
    mse = np.mean((img_orig.astype(np.float32) - img_denoised.astype(np.float32)) ** 2)
    if mse == 0:
        return float('inf')
    psnr = 20 * np.log10(255.0 / np.sqrt(mse))
    return psnr
    """
    Note:
    The PSNR evaluation function was implemented with the help of the AI ​​assistant (DeepSeek).
    Initially, I gave a broad hint, "Write a PSNR function using NumPy," but it resulted in an error.
    After providing the error message to the AI, I obtained the code above.
    """

# Calculate Structural Similarity Index (SSIM)
def calculate_ssim(img_orig, img_denoised):
    return metrics.structural_similarity(img_orig, img_denoised, data_range=255)

**Step2**. Apply two denoising methods to each image and record the metrics

In [ ]:
results = []  # Store results for each image

# Set denoising parameters
dct_quality = 0.8
"""
The DCT quantisation factor was set to `0.8` as a moderate denoising configuration.
Values below 1.0 increase quantisation strength, effectively applying a stronger low-pass filter in the DCT domain.
A factor of `0.8` was chosen empirically to achieve noticeable noise reduction while preserving structural details.
"""
gaussian_sigma = 1.2
"""
The Gaussian filter standard deviation was set to `sigma = 1.2`.
This value was chosen to provide a comparable level of denoising to the DCT method (`quality_factor = 0.8`) for a fair comparison between the two approaches.
"""

for img_path in image_paths:
    # Read as grayscale
    img_orig = io.imread(img_path, as_gray=True)
    if img_orig.dtype != np.uint8:
        img_orig = (img_orig * 255).astype(np.uint8)

    # Denoising
    img_dct = dct_denoise(img_orig, quality_factor=dct_quality)
    img_gauss = gaussian_denoise(img_orig, sigma=gaussian_sigma)

    # Calculate metrics
    psnr_dct = calculate_psnr(img_orig, img_dct)
    ssim_dct = calculate_ssim(img_orig, img_dct)
    psnr_gauss = calculate_psnr(img_orig, img_gauss)
    ssim_gauss = calculate_ssim(img_orig, img_gauss)

    results.append({
        'name': os.path.basename(img_path),
        'psnr_dct': psnr_dct, 'ssim_dct': ssim_dct,
        'psnr_gauss': psnr_gauss, 'ssim_gauss': ssim_gauss,
        'img_orig': img_orig, 'img_dct': img_dct, 'img_gauss': img_gauss
    })
    print(f"{os.path.basename(img_path)}: DCT (PSNR={psnr_dct:.2f}, SSIM={ssim_dct:.3f}) | Gauss (PSNR={psnr_gauss:.2f}, SSIM={ssim_gauss:.3f})")

## **Visualisation**

**Step1**. Examples of the processing results for some of the images

In [ ]:
# Pick the first 4 images as examples
selected_images = results[:4]  # First 4 images from results

# Create 4×3 layout (4 rows: each image; 3 columns: Original, DCT, Gaussian)
fig, axes = plt.subplots(4, 3, figsize=(15, 20))

for idx, sample in enumerate(selected_images):
    titles = [
        f'{sample["name"]}\nOriginal',
        f'DCT Denoised\nPSNR={sample["psnr_dct"]:.1f} dB\nSSIM={sample["ssim_dct"]:.3f}',
        f'Gaussian Denoised\nPSNR={sample["psnr_gauss"]:.1f} dB\nSSIM={sample["ssim_gauss"]:.3f}'
    ]
    images = [sample['img_orig'], sample['img_dct'], sample['img_gauss']]

    for col, (img, title) in enumerate(zip(images, titles)):
        axes[idx, col].imshow(img, cmap='gray')
        axes[idx, col].set_title(title, fontsize=10)
        axes[idx, col].axis('off')

plt.suptitle('Denoising Results Comparison (First 4 Images)', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

**step2**. PSNR/SSIM comparison bar charts for all images

In [ ]:
names = [r['name'] for r in results]
psnr_dct = [r['psnr_dct'] for r in results]
psnr_gauss = [r['psnr_gauss'] for r in results]
ssim_dct = [r['ssim_dct'] for r in results]
ssim_gauss = [r['ssim_gauss'] for r in results]

x = np.arange(len(names))
width = 0.35

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(20, 10))

# PSNR
ax1.bar(x - width/2, psnr_dct, width, label='DCT')
ax1.bar(x + width/2, psnr_gauss, width, label='Gaussian')
ax1.set_ylabel('PSNR (dB)')
ax1.set_title('PSNR Comparison')
ax1.set_xticks(x)
ax1.set_xticklabels(names, rotation=45, ha='right')
ax1.legend()

# SSIM
ax2.bar(x - width/2, ssim_dct, width, label='DCT')
ax2.bar(x + width/2, ssim_gauss, width, label='Gaussian')
ax2.set_ylabel('SSIM')
ax2.set_title('SSIM Comparison')
ax2.set_xticks(x)
ax2.set_xticklabels(names, rotation=45, ha='right')
ax2.legend()

plt.tight_layout()
plt.show()